### Importing packages and modules

In [1]:
# module for building the pyomo model
import pyomo.environ as pe
# module for solving the pyomo model
import pyomo.opt as po

### Create the model

In [2]:
model = pe.ConcreteModel()

Order to build the model:
1. Sets
1. Parameters
1. Variables
1. Objective function
1. Constraints

#### Sets

$d$: demands {'d1', 'd2'}

In [3]:
model.demands = pe.Set(initialize=['d1', 'd2'])

$g$: generators {'g1', 'g2'}

In [4]:
model.generators = pe.Set(initialize=['g1', 'g2'])

#### Parameters

$U_{d}$: bid price of demand d

In [5]:
bid_price_demand = {
    'd1': 40,
    'd2': 35
}

model.bid_price_demand = pe.Param(model.demands, initialize = bid_price_demand)

$C_{g}$: offer price of generator g

In [6]:
offer_price_generator = {
    'g1': 12,
    'g2': 20
}

model.offer_price_generator = pe.Param(model.generators, initialize = offer_price_generator)

$\overline{P_d^D}$: maximum load of demand d

In [7]:
max_load_demand = {
    'd1': 100,
    'd2': 50
}

model.max_load_demand = pe.Param(model.demands, initialize = max_load_demand)

$\overline{P_g^G}$: capacity of generator g

In [8]:
capacity_generator = {
    'g1': 100,
    'g2': 80
}

model.capacity_generator = pe.Param(model.generators, initialize = capacity_generator)

#### Variables

$p_d^D$: quantity purchased by demand d

In [9]:
model.quantity_purchased_demand = pe.Var(model.demands, within = pe.NonNegativeReals)

$p_g^G$: quantity purchased to generator g

In [10]:
model.quantity_purchased_generator = pe.Var(model.generators, within = pe.NonNegativeReals)

#### Objective Function

max $\sum_{d} U_{d} \space p_d^D - \sum_{g} C_{g} \space p_g^G$

In [11]:
def obj_rule(model):
    return sum(model.bid_price_demand[d] * model.quantity_purchased_demand[d] for d in model.demands) - sum(model.offer_price_generator[g] * model.quantity_purchased_generator[g] for g in model.generators)

model.cost = pe.Objective(rule = obj_rule, sense = pe.maximize)

#### Constraints

Constraint #1: maximum load of demand d

$ p_d^D \leq \overline{P_d^D} \quad \forall d$

In [12]:
model.constraint_demand_max_load = pe.ConstraintList()

for d in model.demands:
    model.constraint_demand_max_load.add(
        model.quantity_purchased_demand[d] <= model.max_load_demand[d]
    )

Constraint #2: maximum capacity of generator g

$ p_g^G \leq \overline{P_g^G} \quad \forall d$

In [13]:
model.constraint_generator_capacity = pe.ConstraintList()

for g in model.generators:
    model.constraint_generator_capacity.add(
        model.quantity_purchased_generator[g] <= model.capacity_generator[g]
    )

Constraint #3: ensure balance in demand and production

$ \sum_{d} p_d^D - \sum_{g} p_g^G = 0$

In [14]:
def constraint_balance(model):
    return sum(model.quantity_purchased_demand[d] for d in model.demands) - sum(model.quantity_purchased_generator[g] for g in model.generators) == 0

model.constraint_balance = pe.Constraint(rule=constraint_balance)

#### Solver definition and solve statement

In [ ]:
solver = po.SolverFactory('gurobi')

if not hasattr(model, 'dual'):
	model.dual = pe.Suffix(direction=pe.Suffix.IMPORT)

results = solver.solve(model, tee=True)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-03
Read LP format model from file C:\Users\avela\AppData\Local\Temp\tmprhsb5ztv.pyomo.lp
Reading time = 0.00 seconds
x1: 5 rows, 4 columns, 8 nonzeros
Set parameter QCPDual to value 1
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5 rows, 4 columns and 8 nonzeros
Model fingerprint: 0x2ed7367c
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 4e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+01, 1e+02]
Presolve removed 4 rows and 0 columns
Presolve time: 0.00s
Presolved: 1 rows, 4 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.7500000e+03   1.875000e+01   0.000000e+00    

In [ ]:
dual_val = model.dual.get(model.constraint_balance, None)
if dual_val is not None:
	print(f"Dual value of the balance constraint: {dual_val}")
else:
	print("Dual value of the balance constraint not available.")

data_to_show_demands = {
	"names": ["$U_d$", "$p_d^D$", "$\overline{P_d^D}$"],
	"values": [model.bid_price_demand, model.quantity_purchased_demand, model.max_load_demand],
	"elements": model.demands
}

data_to_show_generators = {
	"names": ["$C_g$", "$p_g^G$", "$\overline{P_g^G}$"],
	"values": [model.offer_price_generator, model.quantity_purchased_generator, model.capacity_generator],
	"elements": model.generators
}

def correct_length(text, length:int, filler=" ", end_extra_fill:bool=True, start_extra_fill:bool=True):
    return f"{filler if start_extra_fill else ''}{filler*(length - len(str(text)))}{text}{filler if end_extra_fill else ''}"

def create_matrix(data_to_show):
	if len(data_to_show["names"]) != len(data_to_show["values"]):
		print("Length of data_to_show['names'] and data_to_show['values'] should be equal.")
		raise ValueError
	matrix = []
	matrix.append(["", *data_to_show["names"]])
	matrix.append(["" for _ in range(len(data_to_show["names"]) + 1)])
	for e in data_to_show["elements"]:
		matrix.append([str(e), *[str(pe.value(data[e])) for data in data_to_show["values"]]])
	for j in range(len(matrix[0])):
		# Get max legth of column
		max_length = max(len(matrix[i][j]) for i in range(len(matrix)))

		# Fill column elements to max length
		for i in range(len(matrix)):
			if i == 1:
				matrix[i][j] = correct_length(matrix[i][j], max_length, filler="-")
			else:
				matrix[i][j] = correct_length(matrix[i][j], max_length)

	return matrix

def show_table(data_to_show):
	matrix = create_matrix(data_to_show)
	print()
	print(" " + "|".join(matrix[0]) + "|")
	print(" " +  " ".join(matrix[1]))
	for i in range(2, len(matrix)):
		print("|" + "|".join(matrix[i]) + "|")

show_table(data_to_show_demands)
show_table(data_to_show_generators)

Dual value of the balance constraint: 20.0

     | $U_d$ | $p_d^D$ | $\overline{P_d^D}$ |
 ---- ------- --------- --------------------
| d1 |    40 |   100.0 |                100 |
| d2 |    35 |    50.0 |                 50 |

     | $C_g$ | $p_g^G$ | $\overline{P_g^G}$ |
 ---- ------- --------- --------------------
| g1 |    12 |   100.0 |                100 |
| g2 |    20 |    50.0 |                 80 |
